# YOLO11m - Colab

Train, evaluate, and run inference with YOLO11m on Google Drive data.
Setup cells mirror `colab_template.ipynb`.

In [1]:
REPO = "road-damage-detection"

# Clone the repository (skip if already cloned)
!test -d $REPO || git clone https://github.com/orzmik/road-damage-detection.git
%cd $REPO

# Install dependencies needed for Colab runs
!pip -q install ultralytics wandb

Cloning into 'road-damage-detection'...
remote: Enumerating objects: 168, done.
remote: Counting objects: 100% (168/168), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 168 (delta 73), reused 136 (delta 45), pack-reused 0 (from 0)
Receiving objects: 100% (168/168), 2.43 MiB | 7.32 MiB/s, done.
Resolving deltas: 100% (73/73), done.
/content/road-damage-detection
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.5 MB/s eta 0:00:00


In [2]:
BRANCH = "feature/yolo_11m-training"

!git fetch origin
!git checkout $BRANCH
!git pull origin $BRANCH

Branch 'feature/yolo_11m-training' set up to track remote branch 'feature/yolo_11m-training' from 'origin'.
Switched to a new branch 'feature/yolo_11m-training'
From https://github.com/orzmik/road-damage-detection
 * branch            feature/yolo_11m-training -> FETCH_HEAD
Already up to date.


In [3]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

In [4]:
# My Drive:
DRIVE_ROOT = "MyDrive/road_damage_detection"

# Shared drive:
# DRIVE_ROOT = "Shareddrives/<TeamDrive>/road_damage_detection"

In [5]:
from src.config.colab.drive import DriveConfig, ensure_drive_paths, mount_drive

mount_drive()

drive_cfg = DriveConfig(drive_root=DRIVE_ROOT)
paths = ensure_drive_paths(drive_cfg)

print(f"Drive root:       {paths.root}")
print(f"Processed data:   {paths.processed_yolo}")
print(f"Models:           {paths.models}")
print(f"Training runs:    {paths.runs}")
print(f"Wroclaw images:   {paths.wroclaw_images}")

Mounted at /content/drive
Drive root:       /content/drive/MyDrive/road_damage_detection
Processed data:   /content/drive/MyDrive/road_damage_detection/data/processed-yolo
Models:           /content/drive/MyDrive/road_damage_detection/models
Training runs:    /content/drive/MyDrive/road_damage_detection/runs
Wroclaw images:   /content/drive/MyDrive/road_damage_detection/wroclaw_images


## Training

In [6]:
import wandb
from google.colab import userdata


wandb.login(key=userdata.get('WANDB_API_KEY'))

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: koostosh (ADM-lists) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [7]:
from src.config.yolo import create_yolo_data_yaml

import shutil
from pathlib import Path

LOCAL_DATA = Path("/content/data/processed-yolo")
DRIVE_DATA = paths.processed_yolo
if not LOCAL_DATA.exists():
    print("Copying dataset from Drive to local disk (one-time per session)...")
    shutil.copytree(DRIVE_DATA, LOCAL_DATA)
    print("Done.")
else:
    print("Local copy already exists, skipping copy.")

data_yaml = create_yolo_data_yaml(
    Path("/content/data/road_damage_local.yaml"),
    LOCAL_DATA,
)

Copying dataset from Drive to local disk (one-time per session)...
Done.


In [8]:
from src.config.wandb import WandbConfig
from src.config.yolo import train_yolo

wandb_cfg = WandbConfig(
    project="road-damage-classification",
    entity="project-nn",
    name="yolo11m_colab",
    job_type="train",
    config={
        "model": "yolo11m",
        "epochs": 50,
        "imgsz": 640,
        "batch": 16,
    },
)

# If CUDA OOM on T4, reduce batch to 8 (keep other hyperparameters unchanged).
train_out = train_yolo(
    weights="yolo11m.pt",
    data_yaml=data_yaml,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=8,
    device=0,
    project_dir=paths.runs,
    run_name="yolo11m_colab",
    wandb_cfg=wandb_cfg,
)

train_out.save_dir

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Ultralytics 8.4.59 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/road_damage_local.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11m_colab, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, p

wandb: WARNING Tried to log to step 50 that is less than the current step 52. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.


lr/pg0,▃▆███▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁
lr/pg1,▃▆███▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁▁
lr/pg2,▃▆███▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁▁
metrics/mAP50(B),▃▂▁▂▃▄▄▄▅▄▅▅▆▅▆▆▆▆▆▆▇▆▆▇▇▇▇▇▇▇██▇█▇█████
metrics/mAP50-95(B),▂▂▁▂▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇█▇██████
metrics/precision(B),▃█▂▁▃▄▅▃▄▄▅▄▄▄▆▆▆▅▆▅▆▅▆▆▆▆▇▆▆▆▇▆▇▆▇▇▇▇▇▇
metrics/recall(B),▃▂▁▂▃▄▄▅▅▄▅▅▆▆▆▆▆▆▆▆▆▇▇▆▇▇▇▇▇▇▇█▇█▇█████
model/GFLOPs,▁
model/parameters,▁
model/speed_PyTorch(ms),▁
+6,...


PosixPath('/content/drive/MyDrive/road_damage_detection/runs/yolo11m_colab')

## Evaluation (test set)

In [9]:
from src.config.yolo import evaluate_yolo

best_weights = train_out.save_dir / "weights" / "best.pt"
metrics = evaluate_yolo(
    weights=best_weights,
    data_yaml=data_yaml,
    split="test",
    device=0,
    project_dir=paths.runs,
    run_name="yolo11m_colab_test",
)
metrics

Ultralytics 8.4.59 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11m summary (fused): 126 layers, 20,032,345 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 23.1±18.2 MB/s, size: 96.3 KB)
val: Scanning /content/data/processed-yolo/test/labels... 202 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 202/202 379.2it/s 0.5s
val: New cache created: /content/data/processed-yolo/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 2.7it/s 4.8s
                   all        202        493      0.537        0.5        0.5      0.212
               Pothole         77        129       0.54      0.364      0.401      0.149
                 Crack        140        272      0.418      0.364      0.342      0.123
               Manhole         70         92      0.654      0.772      0.758      0.364
Speed: 1.5ms preprocess, 17.0ms inference, 0.0ms loss, 1.6ms

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x795691d25b20>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04

## Inference (Wroclaw images)

In [10]:
from src.config.yolo import predict_yolo

predict_results = predict_yolo(
    weights=best_weights,
    source=paths.wroclaw_images,
    device=0,
    project_dir=paths.runs,
    run_name="yolo11m_colab_infer",
    save=True,
)
predict_results


image 1/5 /content/drive/MyDrive/road_damage_detection/wroclaw_images/1779554991904.jpg: 288x640 3 Potholes, 1 Crack, 56.8ms
image 2/5 /content/drive/MyDrive/road_damage_detection/wroclaw_images/1779554991907.jpg: 288x640 2 Potholes, 21.7ms
image 3/5 /content/drive/MyDrive/road_damage_detection/wroclaw_images/1779554991911.jpg: 288x640 1 Pothole, 21.8ms
image 4/5 /content/drive/MyDrive/road_damage_detection/wroclaw_images/1779554991914.jpg: 288x640 3 Potholes, 21.7ms
image 5/5 /content/drive/MyDrive/road_damage_detection/wroclaw_images/1779554991917.jpg: 288x640 (no detections), 21.8ms
Speed: 2.0ms preprocess, 28.8ms inference, 1.8ms postprocess per image at shape (1, 3, 288, 640)
Results saved to /content/drive/MyDrive/road_damage_detection/runs/yolo11m_colab_infer


[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'Pothole', 1: 'Crack', 2: 'Manhole'}
 obb: None
 orig_img: array([[[ 45,  89,  58],
         [ 41,  85,  54],
         [ 40,  84,  53],
         ...,
         [179, 178, 182],
         [179, 178, 182],
         [180, 179, 183]],
 
        [[ 44,  88,  57],
         [ 37,  81,  50],
         [ 36,  80,  49],
         ...,
         [182, 181, 185],
         [183, 182, 186],
         [183, 182, 186]],
 
        [[ 46,  90,  59],
         [ 40,  84,  53],
         [ 39,  83,  52],
         ...,
         [181, 180, 184],
         [179, 178, 182],
         [178, 177, 181]],
 
        ...,
 
        [[125, 127, 128],
         [125, 127, 128],
         [125, 127, 128],
         ...,
         [148, 150, 150],
         [145, 147, 147],
         [146, 148, 148]],
 
        [[127, 129, 130],
         [129, 131, 132],
         [131, 133, 134],
     